# 0. Exploratory Data Analysis

### Purpose

Explore three women's hockey event datasets to understand their coverage,
structure, and quality, and assess whether they support meaningful
comparisons of game pace.

This notebook will examine event patterns, timing, locations, and game
context, and identify the transformations needed to compare the datasets
consistently.

### Questions

#### Data coverage and quality
- Which games, teams, players, and dates does each dataset cover?
- Are there missing values, duplicate records, or inconsistencies?
- Do the recorded events appear to cover complete games?

#### Comparability
- Which columns and event definitions are shared across the datasets?
- How are unsuccessful plays, shots, and goals represented?
- Can time, score, manpower, and coordinates be aligned consistently?

#### Event patterns
- Which events occur most frequently?
- How do event distributions vary across games, teams, and datasets?
- Where on the rink do shots, zone entries, and puck recoveries occur?

### Expected outcome

A documented understanding of each dataset, a proposed common schema,
and initial pace measures with clear assumptions and limitations.

## 0.1. Setup and load raw data

In [70]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [71]:
import pandas as pd
import numpy as np

from pathlib import Path

from utils.hockey_plot import plot_full_rink

pd.set_option('display.max_columns', None)

In [72]:
ROOT_DIR = '..//'
RAW_DATA_DIR = Path(ROOT_DIR) / 'data' / 'raw'

nwhl_df = pd.read_csv(f'{RAW_DATA_DIR}/hackathon_nwhl.csv')
womens_df = pd.read_csv(f'{RAW_DATA_DIR}/hackathon_womens.csv')
olympics_df = pd.read_csv(f'{RAW_DATA_DIR}/pxp_womens_oly_2022_v2.csv')

datasets = {
    'NWHL': nwhl_df,
    'Womens': womens_df,
    'Olympics': olympics_df,
}

In [73]:
nwhl_df.head()

,game_date,Home Team,Away Team,Period,Clock,Home Team Skaters,Away Team Skaters,Home Team Goals,Away Team Goals,Team,Player,Event,X Coordinate,Y Coordinate,Detail 1,Detail 2,Detail 3,Detail 4,Player 2,X Coordinate 2,Y Coordinate 2
0,2021-01-23,Minnesota Whitecaps,Boston Pride,1,20:00,5,5,0,0,Boston Pride,Jillian Dempsey,Faceoff Win,100,43,Backhand,NaN,NaN,NaN,Stephanie Anderson,NaN,NaN
1,2021-01-23,Minnesota Whitecaps,Boston Pride,1,19:58,5,5,0,0,Boston Pride,McKenna Brand,Puck Recovery,107,40,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2021-01-23,Minnesota Whitecaps,Boston Pride,1,19:57,5,5,0,0,Boston Pride,McKenna Brand,Zone Entry,125,28,Carried,NaN,NaN,NaN,Maddie Rowe,NaN,NaN
3,2021-01-23,Minnesota Whitecaps,Boston Pride,1,19:55,5,5,0,0,Boston Pride,McKenna Brand,Shot,131,28,Snapshot,On Net,t,f,NaN,NaN,NaN
4,2021-01-23,Minnesota Whitecaps,Boston Pride,1,19:53,5,5,0,0,Boston Pride,Tereza Vanisova,Faceoff Win,169,21,Backhand,NaN,NaN,NaN,Stephanie Anderson,NaN,NaN


In [74]:
womens_df.head()

,game_date,Home Team,Away Team,Period,Clock,Home Team Skaters,Away Team Skaters,Home Team Goals,Away Team Goals,Team,Player,Event,X Coordinate,Y Coordinate,Detail 1,Detail 2,Detail 3,Detail 4,Player 2,X Coordinate 2,Y Coordinate 2
0,2018-02-11,Olympic (Women) - Canada,Olympic (Women) - Olympic Athletes from Russia,1,20:00,5,5,0,0,Olympic (Women) - Canada,Emily Clark,Faceoff Win,100,42,Backhand,NaN,NaN,NaN,Lyudmila Belyakova,NaN,NaN
1,2018-02-11,Olympic (Women) - Canada,Olympic (Women) - Olympic Athletes from Russia,1,19:59,5,5,0,0,Olympic (Women) - Canada,Meaghan Mikkelson,Puck Recovery,91,67,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2018-02-11,Olympic (Women) - Canada,Olympic (Women) - Olympic Athletes from Russia,1,19:59,5,5,0,0,Olympic (Women) - Canada,Meaghan Mikkelson,Dump In/Out,106,81,Retained,NaN,NaN,NaN,NaN,NaN,NaN
3,2018-02-11,Olympic (Women) - Canada,Olympic (Women) - Olympic Athletes from Russia,1,19:59,5,5,0,0,Olympic (Women) - Canada,Meaghan Mikkelson,Zone Entry,106,81,Dumped,NaN,NaN,NaN,Liana Ganeyeva,NaN,NaN
4,2018-02-11,Olympic (Women) - Canada,Olympic (Women) - Olympic Athletes from Russia,1,19:53,5,5,0,0,Olympic (Women) - Canada,Sarah Nurse,Puck Recovery,168,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [75]:
olympics_df.head()

,game_date,season_year,team_name,opp_team_name,venue,period,clock_seconds,situation_type,goals_for,goals_against,player_name,event,event_successful,x_coord,y_coord,event_type,player_name_2,x_coord_2,y_coord_2,event_detail_1,event_detail_2,event_detail_3
0,8/2/2022,2021,Olympic (Women) - Canada,Olympic (Women) - United States,away,1,1200,5 on 5,0,0,Marie-Philip Poulin,Faceoff Win,t,100,42,Backhand,Hannah Brandt,NaN,NaN,NaN,NaN,NaN
1,8/2/2022,2021,Olympic (Women) - Canada,Olympic (Women) - United States,away,1,1199,5 on 5,0,0,Jocelyne Larocque,Puck Recovery,t,86,31,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,8/2/2022,2021,Olympic (Women) - Canada,Olympic (Women) - United States,away,1,1198,5 on 5,0,0,Jocelyne Larocque,Dump In/Out,f,96,15,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,8/2/2022,2021,Olympic (Women) - Canada,Olympic (Women) - United States,away,1,1197,5 on 5,0,0,Jocelyne Larocque,Zone Entry,t,124,1,Dumped,Cayla Barnes,NaN,NaN,NaN,NaN,NaN
4,8/2/2022,2021,Olympic (Women) - United States,Olympic (Women) - Canada,home,1,1194,5 on 5,0,0,Lee Stecklein,Puck Recovery,t,6,73,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 0.2. Dataset overview

### Shape

In [76]:
for name, df in datasets.items():
   print(f'{name}: {df.shape[0]:,} rows, {df.shape[1]} columns')


NWHL: 26,882 rows, 21 columns
Womens: 24,002 rows, 21 columns
Olympics: 10,609 rows, 22 columns


### Columns

NWHL and NCAA have the same column schema but Olympics is different.

In [77]:
for name, df in datasets.items():
   # Display columns names
   print(f'{name} columns: {df.columns.tolist()}')

NWHL columns: ['game_date', 'Home Team', 'Away Team', 'Period', 'Clock', 'Home Team Skaters', 'Away Team Skaters', 'Home Team Goals', 'Away Team Goals', 'Team', 'Player', 'Event', 'X Coordinate', 'Y Coordinate', 'Detail 1', 'Detail 2', 'Detail 3', 'Detail 4', 'Player 2', 'X Coordinate 2', 'Y Coordinate 2']
Womens columns: ['game_date', 'Home Team', 'Away Team', 'Period', 'Clock', 'Home Team Skaters', 'Away Team Skaters', 'Home Team Goals', 'Away Team Goals', 'Team', 'Player', 'Event', 'X Coordinate', 'Y Coordinate', 'Detail 1', 'Detail 2', 'Detail 3', 'Detail 4', 'Player 2', 'X Coordinate 2', 'Y Coordinate 2']
Olympics columns: ['game_date', 'season_year', 'team_name', 'opp_team_name', 'venue', 'period', 'clock_seconds', 'situation_type', 'goals_for', 'goals_against', 'player_name', 'event', 'event_successful', 'x_coord', 'y_coord', 'event_type', 'player_name_2', 'x_coord_2', 'y_coord_2', 'event_detail_1', 'event_detail_2', 'event_detail_3']


### Games

In [78]:
for name, df in datasets.items():
    if {'Home Team', 'Away Team'}.issubset(df.columns):
        games = df[['game_date', 'Home Team', 'Away Team']].drop_duplicates()
        games = games.rename(columns={'Home Team': 'team_1', 'Away Team': 'team_2'})
    else:
        games = df[['game_date', 'team_name', 'opp_team_name']].drop_duplicates()
        games = games.rename(columns={'team_name': 'team_1', 'opp_team_name': 'team_2'})

    # Order team names consistently so reversed matchups match
    teams = games[['team_1', 'team_2']].apply(lambda row: sorted(row), axis=1)
    games[['team_1', 'team_2']] = pd.DataFrame(teams.tolist(), index=games.index)

    games = games.drop_duplicates().sort_values('game_date')

    print(f'{name}: {len(games)} unique games')
    display(games.reset_index(drop=True))

NWHL: 15 unique games


,game_date,team_1,team_2
0,2021-01-23,Boston Pride,Minnesota Whitecaps
1,2021-01-23,Buffalo Beauts,Connecticut Whale
2,2021-01-23,Metropolitan Riveters,Toronto Six
3,2021-01-24,Boston Pride,Buffalo Beauts
4,2021-01-24,Connecticut Whale,Metropolitan Riveters
5,2021-01-24,Minnesota Whitecaps,Toronto Six
6,2021-01-26,Boston Pride,Toronto Six
7,2021-01-26,Metropolitan Riveters,Minnesota Whitecaps
8,2021-01-27,Boston Pride,Connecticut Whale
9,2021-01-27,Buffalo Beauts,Toronto Six


Womens: 13 unique games


,game_date,team_1,team_2
0,2018-02-11,Olympic (Women) - Canada,Olympic (Women) - Olympic Athletes from Russia
1,2018-02-13,Olympic (Women) - Canada,Olympic (Women) - Finland
2,2018-02-14,Olympic (Women) - Canada,Olympic (Women) - United States
3,2018-02-19,Olympic (Women) - Canada,Olympic (Women) - Olympic Athletes from Russia
4,2018-02-21,Olympic (Women) - Canada,Olympic (Women) - United States
5,2018-10-19,Clarkson Golden Knights,St. Lawrence Saints
6,2018-10-20,Clarkson Golden Knights,St. Lawrence Saints
7,2019-02-12,Olympic (Women) - Canada,Olympic (Women) - United States
8,2019-02-14,Olympic (Women) - Canada,Olympic (Women) - United States
9,2019-02-17,Olympic (Women) - Canada,Olympic (Women) - United States


Olympics: 6 unique games


,game_date,team_1,team_2
0,12/2/2022,Olympic (Women) - Olympic Athletes from Russia,Olympic (Women) - Switzerland
1,14/2/2022,Olympic (Women) - Canada,Olympic (Women) - Switzerland
2,14/2/2022,Olympic (Women) - Finland,Olympic (Women) - United States
3,16/2/2022,Olympic (Women) - Finland,Olympic (Women) - Switzerland
4,8/2/2022,Olympic (Women) - Canada,Olympic (Women) - United States
5,8/2/2022,Olympic (Women) - Finland,Olympic (Women) - Olympic Athletes from Russia


In [79]:
for name, df in datasets.items():
    if {'Home Team', 'Away Team'}.issubset(df.columns):
        game_cols = ['game_date', 'Home Team', 'Away Team']
    else:
        game_cols = ['game_date', 'team_name', 'opp_team_name']

    games = df[game_cols].drop_duplicates().copy()
    games.columns = ['game_date', 'team_1', 'team_2']

    # Make reversed Olympic matchups identical, then keep unique games
    games[['team_1', 'team_2']] = pd.DataFrame(
        games[['team_1', 'team_2']].apply(lambda row: sorted(row), axis=1).tolist(),
        index=games.index,
    )
    games = games.drop_duplicates()

    team_games = pd.concat([games['team_1'], games['team_2']]).value_counts()

    print(f'{name}: games per team')
    display(team_games.rename('games').to_frame())

NWHL: games per team


,games
Boston Pride,7
Buffalo Beauts,6
Toronto Six,6
Connecticut Whale,4
Minnesota Whitecaps,4
Metropolitan Riveters,3


Womens: games per team


,games
Olympic (Women) - Canada,10
Olympic (Women) - United States,7
Olympic (Women) - Finland,3
Clarkson Golden Knights,2
Olympic (Women) - Olympic Athletes from Russia,2
St. Lawrence Saints,2


Olympics: games per team


,games
Olympic (Women) - Finland,3
Olympic (Women) - Switzerland,3
Olympic (Women) - Canada,2
Olympic (Women) - Olympic Athletes from Russia,2
Olympic (Women) - United States,2


## 0.3. Schema comparison

### Schema

In [80]:
for name, df in datasets.items():
    print(f'{name}:\n{list(df.columns)}')
    display()

NWHL:
['game_date', 'Home Team', 'Away Team', 'Period', 'Clock', 'Home Team Skaters', 'Away Team Skaters', 'Home Team Goals', 'Away Team Goals', 'Team', 'Player', 'Event', 'X Coordinate', 'Y Coordinate', 'Detail 1', 'Detail 2', 'Detail 3', 'Detail 4', 'Player 2', 'X Coordinate 2', 'Y Coordinate 2']
Womens:
['game_date', 'Home Team', 'Away Team', 'Period', 'Clock', 'Home Team Skaters', 'Away Team Skaters', 'Home Team Goals', 'Away Team Goals', 'Team', 'Player', 'Event', 'X Coordinate', 'Y Coordinate', 'Detail 1', 'Detail 2', 'Detail 3', 'Detail 4', 'Player 2', 'X Coordinate 2', 'Y Coordinate 2']
Olympics:
['game_date', 'season_year', 'team_name', 'opp_team_name', 'venue', 'period', 'clock_seconds', 'situation_type', 'goals_for', 'goals_against', 'player_name', 'event', 'event_successful', 'x_coord', 'y_coord', 'event_type', 'player_name_2', 'x_coord_2', 'y_coord_2', 'event_detail_1', 'event_detail_2', 'event_detail_3']


In [81]:
field_map = pd.DataFrame([
    ['Game date', 'game_date', 'game_date'],
    ['Season', 'n/a', 'season_year'],
    ['Team', 'Team', 'team_name'],
    ['Opponent', 'Home Team / Away Team', 'opp_team_name'],
    ['Team venue', 'Home Team / Away Team', 'venue'],
    ['Period', 'Period', 'period'],
    ['Clock', 'Clock (MM:SS)', 'clock_seconds'],
    ['Team skaters', 'Home Team Skaters / Away Team Skaters', 'situation_type'],
    ['Team goals', 'Home Team Goals / Away Team Goals', 'goals_for'],
    ['Opponent goals', 'Away Team Goals / Home Team Goals', 'goals_against'],
    ['Player', 'Player', 'player_name'],
    ['Event', 'Event', 'event'],
    ['Event outcome', 'n/a', 'event_successful'],
    ['X coordinate', 'X Coordinate', 'x_coord'],
    ['Y coordinate', 'Y Coordinate', 'y_coord'],
    ['Second player', 'Player 2', 'player_name_2'],
    ['Second player X coordinate', 'X Coordinate 2', 'x_coord_2'],
    ['Second player Y coordinate', 'Y Coordinate 2', 'y_coord_2'],
    ['Event details', 'Detail 1-4 ', 'event_type and event_detail_1-3'],
], columns=['Concept', 'NWHL and Womens', 'Olympics'])

display(field_map)

,Concept,NWHL and Womens,Olympics
0,Game date,game_date,game_date
1,Season,n/a,season_year
2,Team,Team,team_name
3,Opponent,Home Team / Away Team,opp_team_name
4,Team venue,Home Team / Away Team,venue
5,Period,Period,period
6,Clock,Clock (MM:SS),clock_seconds
7,Team skaters,Home Team Skaters / Away Team Skaters,situation_type
8,Team goals,Home Team Goals / Away Team Goals,goals_for
9,Opponent goals,Away Team Goals / Home Team Goals,goals_against


## 0.4. Data quality and consistency

### Duplicates

Only one duplicate which is a double minor, this is fine

In [82]:
for name, df in datasets.items():
    duplicate_count = df.duplicated().sum()
    print(f'{name}: {duplicate_count} duplicate rows')

NWHL: 0 duplicate rows
Womens: 1 duplicate rows
Olympics: 0 duplicate rows


In [83]:
for name, df in datasets.items():
    duplicates = df[df.duplicated(keep=False)]
    print(f'{name}: {len(duplicates)} rows involved in duplicates')
    if not duplicates.empty:
        display(duplicates.head())

NWHL: 0 rows involved in duplicates
Womens: 2 rows involved in duplicates


,game_date,Home Team,Away Team,Period,Clock,Home Team Skaters,Away Team Skaters,Home Team Goals,Away Team Goals,Team,Player,Event,X Coordinate,Y Coordinate,Detail 1,Detail 2,Detail 3,Detail 4,Player 2,X Coordinate 2,Y Coordinate 2
5346,2018-02-14,Olympic (Women) - United States,Olympic (Women) - Canada,3,0:00,6,5,1,2,Olympic (Women) - United States,Jocelyne Lamoureux,Penalty Taken,179,43,Roughing,NaN,NaN,NaN,Marie-Philip Poulin,NaN,NaN
5347,2018-02-14,Olympic (Women) - United States,Olympic (Women) - Canada,3,0:00,6,5,1,2,Olympic (Women) - United States,Jocelyne Lamoureux,Penalty Taken,179,43,Roughing,NaN,NaN,NaN,Marie-Philip Poulin,NaN,NaN


Olympics: 0 rows involved in duplicates


### event_type

_event_type_ in the Olympics dataset is just _Detail 1_ in the others

In [84]:
olympics_df['event_type'].value_counts(dropna=False).rename('count').to_frame()

,count
event_type,
NaN,4388
Direct,2918
Indirect,1474
Carried,420
Snapshot,354
Backhand,287
Dumped,269
Wristshot,235
Slapshot,71


In [85]:
nwhl_df['Detail 1'].value_counts(dropna=False).rename('count').to_frame()

,count
Detail 1,
NaN,9421
Direct,6720
Indirect,3959
Lost,1637
Carried,1267
Snapshot,888
Wristshot,718
Backhand,618
Dumped,578


### Event

Olympics has no Goal and Incomplete Play events. They are embedded in Shot or Play

In [86]:
event_counts = {}

for name, df in datasets.items():
    event_col = 'Event' if 'Event' in df.columns else 'event'
    event_counts[name] = df[event_col].value_counts(dropna=False)

event_counts_df = pd.DataFrame(event_counts).fillna(0).astype(int)
event_counts_df.index.name = 'Event type'

display(event_counts_df.sort_index())

,NWHL,Womens,Olympics
Event type,,,
Dump In/Out,1863,1682,844
Faceoff Win,846,783,335
Goal,76,56,0
Incomplete Play,3430,2681,0
Penalty Taken,144,116,42
Play,7249,7424,4392
Puck Recovery,8214,6960,3229
Shot,1909,1615,709
Takeaway,1207,885,315


In [87]:
for name, df in datasets.items():
    if {'Home Team', 'Away Team'}.issubset(df.columns):
        team_cols = ['Home Team', 'Away Team']
    else:
        team_cols = ['team_name', 'opp_team_name']

    game_rows = df[['game_date', *team_cols]].copy()
    game_rows[['team_1', 'team_2']] = pd.DataFrame(
        game_rows[team_cols].apply(lambda row: sorted(row), axis=1).tolist(),
        index=game_rows.index,
    )

    game_rows['game_id'] = (
        game_rows['game_date'].astype(str) + '|'
        + game_rows['team_1'] + '|' + game_rows['team_2']
    )

    avg_events = game_rows.groupby('game_id').size().mean()
    print(f'{name}: {avg_events:.0f} event rows per game on average')

NWHL: 1792 event rows per game on average
Womens: 1846 event rows per game on average
Olympics: 1768 event rows per game on average


### Missing values by event type

In [88]:
for name, df in datasets.items():
    event_col = 'Event' if 'Event' in df.columns else 'event'

    missing_by_event = (
        df.set_index(event_col)
          .isna()
          .groupby(level=0, dropna=False)
          .sum()
          .astype(int)
    )

    print(f'{name}: missing values by event type')
    display(missing_by_event)

NWHL: missing values by event type


,game_date,Home Team,Away Team,Period,Clock,Home Team Skaters,Away Team Skaters,Home Team Goals,Away Team Goals,Team,Player,X Coordinate,Y Coordinate,Detail 1,Detail 2,Detail 3,Detail 4,Player 2,X Coordinate 2,Y Coordinate 2
Event,,,,,,,,,,,,,,,,,,,,
Dump In/Out,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1863,1863,1863,1863,1863,1863
Faceoff Win,0,0,0,0,0,0,0,0,0,0,0,0,0,0,846,846,846,0,846,846
Goal,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,76,76,76
Incomplete Play,0,0,0,0,0,0,0,0,0,0,0,0,0,0,3430,3430,3430,0,0,0
Penalty Taken,0,0,0,0,0,0,0,0,0,0,0,0,0,0,144,144,144,8,144,144
Play,0,0,0,0,0,0,0,0,0,0,0,0,0,0,7249,7249,7249,0,0,0
Puck Recovery,0,0,0,0,0,0,0,0,0,0,0,0,0,8214,8214,8214,8214,8214,8214,8214
Shot,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1909,1909,1909
Takeaway,0,0,0,0,0,0,0,0,0,0,0,0,0,1207,1207,1207,1207,1207,1207,1207


Womens: missing values by event type


,game_date,Home Team,Away Team,Period,Clock,Home Team Skaters,Away Team Skaters,Home Team Goals,Away Team Goals,Team,Player,X Coordinate,Y Coordinate,Detail 1,Detail 2,Detail 3,Detail 4,Player 2,X Coordinate 2,Y Coordinate 2
Event,,,,,,,,,,,,,,,,,,,,
Dump In/Out,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1682,1682,1682,1682,1682,1682
Faceoff Win,0,0,0,0,0,0,0,0,0,0,0,0,0,0,783,783,783,0,783,783
Goal,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,56,56,56
Incomplete Play,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2681,2681,2681,0,0,0
Penalty Taken,0,0,0,0,0,0,0,0,0,0,0,0,0,0,116,116,116,4,116,116
Play,0,0,0,0,0,0,0,0,0,0,0,0,0,0,7424,7424,7424,0,0,0
Puck Recovery,0,0,0,0,0,0,0,0,0,0,0,0,0,6960,6960,6960,6960,6960,6960,6960
Shot,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1615,1615,1615
Takeaway,0,0,0,0,0,0,0,0,0,0,0,0,0,885,885,885,885,885,885,885


Olympics: missing values by event type


,game_date,season_year,team_name,opp_team_name,venue,period,clock_seconds,situation_type,goals_for,goals_against,player_name,event_successful,x_coord,y_coord,event_type,player_name_2,x_coord_2,y_coord_2,event_detail_1,event_detail_2,event_detail_3
event,,,,,,,,,,,,,,,,,,,,,
Dump In/Out,0,0,0,0,0,0,0,0,0,0,0,0,0,0,844,844,844,844,844,844,844
Faceoff Win,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,335,335,335,335,335
Penalty Taken,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,42,42,42,42,42
Play,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,4392,4392,4392
Puck Recovery,0,0,0,0,0,0,0,0,0,0,0,0,0,0,3229,3229,3229,3229,3229,3229,3229
Shot,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,709,709,709,0,0,0
Takeaway,0,0,0,0,0,0,0,0,0,0,0,0,0,0,315,315,315,315,315,315,315
Zone Entry,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,743,743,743,743,743


### Numeric ranges

In [89]:
for name, df in datasets.items():
    print(f'{name}: numeric ranges')
    display(df.select_dtypes(include='number').describe().T[['min', 'max']])

NWHL: numeric ranges


,min,max
Period,1.0,4.0
Home Team Skaters,3.0,6.0
Away Team Skaters,3.0,6.0
Home Team Goals,0.0,6.0
Away Team Goals,0.0,7.0
X Coordinate,0.0,200.0
Y Coordinate,0.0,85.0
X Coordinate 2,0.0,200.0
Y Coordinate 2,0.0,85.0


Womens: numeric ranges


,min,max
Period,1.0,4.0
Home Team Skaters,3.0,6.0
Away Team Skaters,3.0,6.0
Home Team Goals,0.0,5.0
Away Team Goals,0.0,5.0
X Coordinate,0.0,200.0
Y Coordinate,0.0,85.0
X Coordinate 2,0.0,200.0
Y Coordinate 2,0.0,85.0


Olympics: numeric ranges


,min,max
season_year,2021.0,2021.0
period,1.0,3.0
clock_seconds,1.0,1200.0
goals_for,0.0,10.0
goals_against,0.0,10.0
x_coord,0.0,200.0
y_coord,0.0,85.0
x_coord_2,0.0,200.0
y_coord_2,0.0,85.0


## 0.5. Spatial exploration

### Shots

In [90]:
for name, df in datasets.items():
    print(f'{name} shot locations')
    if name in ['NWHL', 'Womens']:
        shots = df.rename(
            columns={'X Coordinate': 'x', 'Y Coordinate': 'y', 'Event': 'event'}
        )
    else:
        shots = df.rename(
            columns={'x_coord': 'x', 'y_coord': 'y'}
        )

    plot_full_rink(shots, events='Shot').show()

NWHL shot locations

Womens shot locations


Olympics shot locations


### Play

In [91]:
for name, df in datasets.items():
    print(f'{name} passes locations')
    if name in ['NWHL', 'Womens']:
        shots = df.rename(
            columns={'X Coordinate': 'x', 'Y Coordinate': 'y', 'Event': 'event'}
        )
    else:
        shots = df.rename(
            columns={'x_coord': 'x', 'y_coord': 'y'}
        )

    plot_full_rink(shots, events='Play').show()

NWHL passes locations


Womens passes locations


Olympics passes locations


### Faceoffs

In [92]:
for name, df in datasets.items():
    print(f'{name} faceoffs locations')
    if name in ['NWHL', 'Womens']:
        shots = df.rename(
            columns={'X Coordinate': 'x', 'Y Coordinate': 'y', 'Event': 'event'}
        )
    else:
        shots = df.rename(
            columns={'x_coord': 'x', 'y_coord': 'y'}
        )

    plot_full_rink(shots, events='Faceoff Win').show()

NWHL faceoffs locations


Womens faceoffs locations


Olympics faceoffs locations


### Entries

In [93]:
for name, df in datasets.items():
    print(f'{name} entries locations')
    if name in ['NWHL', 'Womens']:
        shots = df.rename(
            columns={'X Coordinate': 'x', 'Y Coordinate': 'y', 'Event': 'event'}
        )
    else:
        shots = df.rename(
            columns={'x_coord': 'x', 'y_coord': 'y'}
        )

    plot_full_rink(shots, events='Zone Entry').show()

NWHL entries locations


Womens entries locations


Olympics entries locations


In [94]:
# Explore why zone entries are mostly aligned with the blue line in the Olympics dataset, but not in the NWHL or Womens datasets.
# Print 5 entries from each dataset

for name, df in datasets.items():
    if name in ['NWHL', 'Womens']:
        entries = df[df['Event'] == 'Zone Entry'].rename(
            columns={'X Coordinate': 'x', 'Y Coordinate': 'y', 'Event': 'event'}
        )
    else:
        entries = df[df['event'] == 'Zone Entry'].rename(
            columns={'x_coord': 'x', 'y_coord': 'y'}
        )

    print(f'{name} Zone Entry events:')
    display(entries.head())


NWHL Zone Entry events:


,game_date,Home Team,Away Team,Period,Clock,Home Team Skaters,Away Team Skaters,Home Team Goals,Away Team Goals,Team,Player,event,x,y,Detail 1,Detail 2,Detail 3,Detail 4,Player 2,X Coordinate 2,Y Coordinate 2
2,2021-01-23,Minnesota Whitecaps,Boston Pride,1,19:57,5,5,0,0,Boston Pride,McKenna Brand,Zone Entry,125,28,Carried,NaN,NaN,NaN,Maddie Rowe,NaN,NaN
7,2021-01-23,Minnesota Whitecaps,Boston Pride,1,19:48,5,5,0,0,Minnesota Whitecaps,Stephanie Anderson,Zone Entry,125,6,Carried,NaN,NaN,NaN,Kaleigh Fratkin,NaN,NaN
19,2021-01-23,Minnesota Whitecaps,Boston Pride,1,19:23,5,5,0,0,Minnesota Whitecaps,Allie Thunstrom,Zone Entry,124,2,Carried,NaN,NaN,NaN,Kaleigh Fratkin,NaN,NaN
27,2021-01-23,Minnesota Whitecaps,Boston Pride,1,19:09,5,5,0,0,Minnesota Whitecaps,Nina Rodgers,Zone Entry,124,20,Played,NaN,NaN,NaN,Taylor Turnquist,NaN,NaN
41,2021-01-23,Minnesota Whitecaps,Boston Pride,1,18:37,5,5,0,0,Boston Pride,Jillian Dempsey,Zone Entry,125,70,Carried,NaN,NaN,NaN,Rose Alleva,NaN,NaN


Womens Zone Entry events:


,game_date,Home Team,Away Team,Period,Clock,Home Team Skaters,Away Team Skaters,Home Team Goals,Away Team Goals,Team,Player,event,x,y,Detail 1,Detail 2,Detail 3,Detail 4,Player 2,X Coordinate 2,Y Coordinate 2
3,2018-02-11,Olympic (Women) - Canada,Olympic (Women) - Olympic Athletes from Russia,1,19:59,5,5,0,0,Olympic (Women) - Canada,Meaghan Mikkelson,Zone Entry,106,81,Dumped,NaN,NaN,NaN,Liana Ganeyeva,NaN,NaN
10,2018-02-11,Olympic (Women) - Canada,Olympic (Women) - Olympic Athletes from Russia,1,19:37,5,5,0,0,Olympic (Women) - Olympic Athletes from Russia,Valeria Pavlova,Zone Entry,124,2,Carried,NaN,NaN,NaN,Meaghan Mikkelson,NaN,NaN
15,2018-02-11,Olympic (Women) - Canada,Olympic (Women) - Olympic Athletes from Russia,1,19:27,5,5,0,0,Olympic (Women) - Canada,Sarah Nurse,Zone Entry,97,3,Dumped,NaN,NaN,NaN,Angelina Goncharenko,NaN,NaN
21,2018-02-11,Olympic (Women) - Canada,Olympic (Women) - Olympic Athletes from Russia,1,19:18,5,5,0,0,Olympic (Women) - Canada,Melodie Daoust,Zone Entry,125,31,Played,NaN,NaN,NaN,Angelina Goncharenko,NaN,NaN
28,2018-02-11,Olympic (Women) - Canada,Olympic (Women) - Olympic Athletes from Russia,1,19:06,5,5,0,0,Olympic (Women) - Canada,Marie-Philip Poulin,Zone Entry,125,79,Carried,NaN,NaN,NaN,Angelina Goncharenko,NaN,NaN


Olympics Zone Entry events:


,game_date,season_year,team_name,opp_team_name,venue,period,clock_seconds,situation_type,goals_for,goals_against,player_name,event,event_successful,x,y,event_type,player_name_2,x_coord_2,y_coord_2,event_detail_1,event_detail_2,event_detail_3
3,8/2/2022,2021,Olympic (Women) - Canada,Olympic (Women) - United States,away,1,1197,5 on 5,0,0,Jocelyne Larocque,Zone Entry,t,124,1,Dumped,Cayla Barnes,NaN,NaN,NaN,NaN,NaN
24,8/2/2022,2021,Olympic (Women) - United States,Olympic (Women) - Canada,home,1,1156,5 on 5,0,0,Hilary Knight,Zone Entry,f,126,2,Dumped,Renata Fast,NaN,NaN,NaN,NaN,NaN
35,8/2/2022,2021,Olympic (Women) - Canada,Olympic (Women) - United States,away,1,1136,5 on 5,0,0,Jamie Lee Rattray,Zone Entry,f,124,57,Dumped,Savannah Harmon,NaN,NaN,NaN,NaN,NaN
50,8/2/2022,2021,Olympic (Women) - United States,Olympic (Women) - Canada,home,1,1117,5 on 5,0,0,Lee Stecklein,Zone Entry,f,125,30,Dumped,Ashton Bell,NaN,NaN,NaN,NaN,NaN
76,8/2/2022,2021,Olympic (Women) - Canada,Olympic (Women) - United States,away,1,1069,5 on 5,0,0,Sarah Nurse,Zone Entry,t,124,8,Dumped,Cayla Barnes,NaN,NaN,NaN,NaN,NaN


## 0.6. Final schema

Proposed SQL table schema

The schema uses one row per game in `games`, one row per recorded play-by-play event in `events`, and one row per distinct team or player in their respective tables.

### `teams`

One row per team.

| Column | SQL type | Nullable | Description |
|---|---|---:|---|
| `team_id` | `INTEGER` | No | Primary key |
| `team_name` | `VARCHAR(100)` | No | Team name as recorded in the source |
| `source_dataset` | `VARCHAR(30)` | No | Dataset the team record came from |

### `players`

One row per distinct player name and team within a source dataset.

| Column | Suggested SQL type | Nullable | Description |
|---|---|---:|---|
| `player_id` | `INTEGER` | No | Primary key |
| `player_name` | `VARCHAR(100)` | No | Player name as recorded in the source |
| `team_id` | `INTEGER` | No | Foreign key to `teams` |
| `source_dataset` | `VARCHAR(30)` | No | Dataset the player record came from |

### `games`

One row per unique game. Home and away teams are nullable because the Olympics file records a team and its opponent with a venue field.

| Column | SQL type | Nullable | Description |
|---|---|---:|---|
| `game_id` | `INTEGER` | No | Primary key |
| `game_date` | `DATE` | No | Game date |
| `home_team_id` | `INTEGER` | No | Foreign key to `teams`; populate when home team is known |
| `away_team_id` | `INTEGER` | No | Foreign key to `teams`; populate when away team is known |
| `source_dataset` | `VARCHAR(30)` | No | Dataset the game came from |

### `events`

One row per recorded event. Event-specific details and secondary player/location fields are nullable because they only apply to some event types.

| Column | Suggested SQL type | Nullable | Description |
|---|---|---:|---|
| `event_id` | `BIGINT` | No | Primary key |
| `game_id` | `INTEGER` | No | Foreign key to `games` |
| `team_id` | `INTEGER` | No | Foreign key to `teams`; team credited with the event |
| `opponent_team_id` | `INTEGER` | No | Foreign key to `teams`; opponent of the team credited with the event |
| `player_id` | `INTEGER` | Yes | Foreign key to `players`; player credited with the event |
| `player_2_id` | `INTEGER` | Yes | Foreign key to `players`; second player involved |
| `period` | `SMALLINT` | No | Period number |
| `clock_seconds` | `DECIMAL(6,10)` | Yes | Time remaining in period, normalized to seconds |
| `team_goals` | `SMALLINT` | Yes | Event team’s score at event time |
| `opponent_goals` | `SMALLINT` | Yes | Opponent’s score at event time |
| `team_skaters` | `SMALLINT` | No | Number of skaters on ice, for the team credited for the event |
| `opponent_skaters` | `SMALLINT` | No | Number of skaters on ice, for the team not credited for the event |
| `event` | `VARCHAR(40)` | No | Recorded broad event label |
| `x` | `DECIMAL(6,2)` | Yes | Primary event x coordinate on the 200×85 rink |
| `y` | `DECIMAL(6,2)` | Yes | Primary event y coordinate on the 200×85 rink |
| `detail_1` | `VARCHAR(100)` | Yes | First source event detail |
| `detail_2` | `VARCHAR(100)` | Yes | Second source event detail |
| `detail_3` | `VARCHAR(100)` | Yes | Third source event detail |
| `detail_4` | `VARCHAR(100)` | Yes | Fourth source event detail |
| `is_home` | `BOOLEAN` | No | True when the team credited with the event is the home team |
| `source_dataset` | `VARCHAR(30)` | No | Dataset the game came from |